# YOLO One-Class Cow Detector Training

**Objective**: Train YOLO11n detector for single-class cow detection using VIA annotation data.

**Dataset**: 25K+ bounding box annotations from VIA CSV across 537 video sequences  
**Model**: YOLO11n (nano) - fast and efficient for detection
**Strategy**: Video-based splitting to prevent data leakage

**Pipeline**: VIA CSV → YOLO Format → Train/Val/Test Split → Model Training → Validation

In [1]:
# Core Python & data handling
import json, ast, os, shutil
from pathlib import Path
from collections import defaultdict
from random import Random

# Data processing
import pandas as pd
import cv2
import yaml

# ML libraries
import torch
from ultralytics import YOLO

# Set seed for reproducibility
torch.manual_seed(42)

In [2]:
# Configuration
CSV_PATH = Path("data/CBVD-5.csv")
IMG_ROOT = Path("data/labelframes/labelframes") 
OUT_ROOT = Path("workdir/yolo_cow_oneclass")

# Training parameters
EPOCHS = 30
IMG_SIZE = 640
MODEL = "yolo11n.pt"  # Updated to YOLO11 nano
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.2
TEST_SPLIT = 0.1
SEED = 42

print(f"Dataset: {CSV_PATH} → {OUT_ROOT}")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

Dataset: data/CBVD-5.csv → workdir/yolo_cow_oneclass


In [3]:
# Helper functions
def parse_file_list(s):
    """Parse VIA file list format"""
    if isinstance(s, list): return s
    try: return ast.literal_eval(s)
    except Exception: return [s]

def parse_box(spatial_coordinates):
    """Extract bounding box from VIA format"""
    coords = json.loads(spatial_coordinates) if isinstance(spatial_coordinates, str) else spatial_coordinates
    if isinstance(coords[0], list): coords = coords[0]
    _, x, y, w, h = coords
    return float(x), float(y), float(w), float(h)

def video_id_from_name(name):
    """Extract video ID from filename (e.g., '618_00002.jpg' → '618')"""
    stem = Path(name).stem
    return stem.split("_")[0] if "_" in stem else stem

def to_yolo_norm(x, y, w, h, W, H):
    """Convert VIA box to YOLO normalized format"""
    cx, cy = (x + w/2.0) / W, (y + h/2.0) / H
    return cx, cy, w / W, h / H

In [4]:
# Load VIA data and create video-based splits
df = pd.read_csv(CSV_PATH, skiprows=9)
assert {"file_list", "spatial_coordinates"}.issubset(set(df.columns))

# Parse annotations into boxes per image
boxes_by_image = defaultdict(list)
for _, row in df.iterrows():
    files = parse_file_list(row["file_list"])
    if not files: continue
    img_name = files[0]
    x, y, w, h = parse_box(row["spatial_coordinates"])
    boxes_by_image[img_name].append((x, y, w, h))

# Create video-based splits to prevent leakage
rng = Random(SEED)
vids = sorted({video_id_from_name(n) for n in boxes_by_image.keys()})
rng.shuffle(vids)

n = len(vids)
n_train = int(n * TRAIN_SPLIT)
n_val = int(n * VAL_SPLIT)

vid_splits = {
    "train": set(vids[:n_train]),
    "val": set(vids[n_train:n_train+n_val]), 
    "test": set(vids[n_train+n_val:])
}

def get_split(img_name):
    vid = video_id_from_name(img_name)
    for split, vid_set in vid_splits.items():
        if vid in vid_set: return split
    return "test"

print(f"Videos: {len(vid_splits['train'])} train, {len(vid_splits['val'])} val, {len(vid_splits['test'])} test")
print(f"Images: {len(boxes_by_image)} with annotations")

Videos: 375 train, 107 val, 55 test
Images: 3199 with annotations


In [5]:
# Create YOLO dataset structure
for split in ["train", "val", "test"]:
    (OUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

# Process images and create YOLO labels
copied, written, missing = 0, 0, 0
for img_name, boxes in boxes_by_image.items():
    src = IMG_ROOT / img_name
    if not src.exists():
        missing += 1
        continue
    
    img = cv2.imread(str(src))
    if img is None:
        missing += 1
        continue
    
    H, W = img.shape[:2]
    split = get_split(img_name)
    
    # Copy image
    dst_img = OUT_ROOT / "images" / split / img_name
    shutil.copy2(src, dst_img)
    copied += 1
    
    # Create YOLO label file
    yolo_lines = []
    for x, y, w, h in boxes:
        cx, cy, nw, nh = to_yolo_norm(x, y, w, h, W, H)
        cx = max(0, min(1, cx)); cy = max(0, min(1, cy))
        nw = max(1e-6, min(1, nw)); nh = max(1e-6, min(1, nh))
        yolo_lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
    
    dst_lbl = OUT_ROOT / "labels" / split / (Path(img_name).stem + ".txt")
    with open(dst_lbl, "w") as f:
        f.write("\n".join(yolo_lines))
    written += 1

# Create data.yaml
data_yaml = {
    "path": str(OUT_ROOT.resolve()),
    "train": "images/train", "val": "images/val", "test": "images/test",
    "names": {0: "cow"}, "nc": 1
}
with open(OUT_ROOT / "data.yaml", "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"Processed: {copied} images, {written} labels, {missing} missing")

Processed: 3199 images, 3199 labels, 0 missing


In [ ]:
# Train YOLO model
# Device selection with fallback for CUDA, MPS, or CPU
if torch.cuda.is_available():
    device = "cuda"
    print(f"Training on: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Training on: Apple Silicon GPU (MPS)")
else:
    device = "cpu"
    print("Training on: CPU")

model = YOLO(MODEL)
results = model.train(
    data=str(OUT_ROOT / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    device=device,
    verbose=False
)

print(f"Training complete. Model saved to: runs/detect/train*/weights/best.pt")

Training on: CPU
Ultralytics 8.3.204 🚀 Python-3.12.11 torch-2.8.0 CPU (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=workdir/yolo_cow_oneclass/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, p

In [ ]:
# Validation and demo
try:
    # Find trained model
    model_paths = list(Path("runs/detect").rglob("weights/best.pt"))
    if not model_paths:
        print("No trained model found. Run training first.")
    else:
        best_model = str(model_paths[-1])  # Use latest
        detector = YOLO(best_model)
        
        # Test on validation images
        val_imgs = list((OUT_ROOT / "images" / "val").glob("*.jpg"))[:3]
        if val_imgs:
            total_detections = 0
            for img_path in val_imgs:
                results = detector.predict(source=str(img_path), conf=0.25, verbose=False)[0]
                detections = len(results.boxes) if results.boxes is not None else 0
                total_detections += detections
                print(f"{img_path.name}: {detections} cows detected")
            
            print(f"✓ Validation: {total_detections} total detections on {len(val_imgs)} images")
            print(f"Model ready at: {best_model}")
        else:
            print("No validation images found")

except Exception as e:
    print(f"Validation error: {e}")
    print("Check training completion and model paths")

85_00007.jpg: 7 cows detected
224_00005.jpg: 17 cows detected
379_00002.jpg: 7 cows detected
✓ Validation: 31 total detections on 3 images
Model ready at: runs/detect/train3/weights/best.pt
